# Chapter 1 — memorisation vs relational knowledge

**Settings:** GPU **T4 ×2** · Internet **ON** · Persistence **Variables and Files**

> Nothing to attach. Section 1 downloads WN11 and FB13 from KG-LLM's repo.

---

### The claim

> Fine-tuning for KGC installs **entity-name memorisation**, not relational structure.
> Measured so far: memorisation **0.393** of the 0.4315 above-chance gain — **91%**.

### Three rules learned the hard way

**1 · One GPU per job.** Two visible → `DataParallel` → autocast never reaches the replicas → `mat1 and mat2 must have the same dtype`. Every command below is pinned.

**2 · fp16 + `sdpa`.** fp16 + `eager` returns **NaN** on Qwen2.5 — it looks like `train_loss=0.0` with `grad_norm=nan`, i.e. a finished run.

**3 · Remove `torchao`.** Kaggle ships 0.10.0; transformers refuses to import below 0.16. Nothing here uses it.

### This chapter needs only LoRA — no MoRA fork, no BOFT kernel

So it runs in **one** session with official peft and none of the environment conflicts that blocked Chapters 2 and 3.

## 0 · Setup

In [ ]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
DEST = "/kaggle/working/repo"

import os, subprocess, sys, socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No network. Settings > Internet > ON, then re-run.")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, DEST])
    print("cloned")

os.chdir(DEST); sys.path.insert(0, DEST)
print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("\n\u2605 Does that hash match your latest push?")

In [ ]:
PIN_TRANSFORMERS = "4.57.6"

import json, subprocess, sys
def stack():
    code = ("import json, peft, transformers; print(json.dumps({"
            "'peft': peft.__version__, 'tf': transformers.__version__}))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if not (s and s["tf"] == PIN_TRANSFORMERS):
    print(f"installing… (have {s})")
    !pip install -q -r requirements.txt
    !pip install -q peft "transformers=={PIN_TRANSFORMERS}"

# torchao 0.10.0 blocks the transformers import and kills LoRA. Nothing uses it.
!pip uninstall -y -q torchao 2>/dev/null

import peft, transformers
print(f"\npeft {peft.__version__} | transformers {transformers.__version__}")

## 1 · Tests — 2 seconds, no GPU

Each test is a worked example: it prints the actual prompt each variant produces. Read the output — it is the clearest documentation of what P0–P4 mean.

In [ ]:
!python -m chapter1.test_chapter1

In [ ]:
# the grid, the pre-registered interpretations, and the cost estimate
!python -m chapter1.conditions
!python -m chapter1.run --plan

## 2 · Data

**WN11** carries the memorisation result. **FB13** carries the type-rule result — its relations (`profession`, `nationality`, `place_of_birth`) have real domain/range, whereas WN11's "types" are only parts of speech. `Person –bornIn→ Location` is a Freebase statement, not a WordNet one.

In [ ]:
!python -m scripts.fetch_data --datasets WN11 FB13

# the plain + anonymised test sets that every `gap` measurement needs
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42 --anonymise

In [ ]:
# build every condition. Prints one example prompt per condition — check them.
!python -m chapter1.data --all --dataset WN11

## 3 · Train

**C and G first — they carry the claim.** C asks whether types can substitute for names; G asks whether types help at all when names are present, which is what CATS, Knit and RealKGC assert and none isolates.

A and B may already exist from the earlier run (0.9315 / 0.5385).

In [ ]:
import subprocess
def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

R = "python -m chapter1.run --dataset WN11 --train --condition"
pair(f"{R} C", f"{R} G")

In [ ]:
pair(f"{R} D", f"{R} E")     # negative hardness, then negative count

In [ ]:
# only if A and B are not already on disk
# pair(f"{R} A", f"{R} B")

## 4 · Evaluate — both test sets, always

Each model is scored on the **real** and the **anonymised** test set. The **gap** is the result; a single accuracy number cannot express the claim.

This also emits the seen/unseen split and the calibration-by-familiarity block for free.

In [ ]:
for C in ["A", "B", "C", "D", "E", "G"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.run --dataset WN11 --evaluate --condition {C}

In [ ]:
!python -m chapter1.analysis --dataset WN11

## 5 · Link prediction — is this really KGC?

Triple classification completes nothing. Here the classifier becomes a **ranker**: score every candidate tail by `P(Yes | h, r, t)` and sort. No retraining — the model was trained to emit exactly that judgement.

★ **This makes MRR computable**, which the spec had recorded as impossible under generative decoding.

⚠️ **50-way, filtered.** Not comparable to full-ranking numbers (R12). Say so in every caption.

In [ ]:
for C in ["A", "B", "C"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
        --adapter checkpoints/ch1-WN11-{C} --dataset WN11 --condition {C} --limit 500

## 6 · Prompt ablation — inference only

Same checkpoints, only the evaluation prompt changes.

⚠️ **P2 and P3 are inert on the tuned model.** Loss was masked to the response, and the response is the fixed string `"Yes, this is true."` — step-by-step instruction-following was tuned *out*. Only the **untuned** arm can react to a reasoning instruction, and that contrast is itself a finding.

⚠️ The tuned model saw only P0 in training, so a **drop** under P1–P4 may be distribution shift. **Only increases are interpretable.**

In [ ]:
# untuned: all five variants are meaningful
for P in ["P0", "P1", "P2", "P3", "P4"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank --dataset WN11 \
        --condition B --prompt {P} --limit 300 --tag ch1prompt-untuned-{P}

In [ ]:
# tuned: only P0, P1, P4 are interpretable
for P in ["P0", "P1", "P4"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank --dataset WN11 \
        --adapter checkpoints/ch1-WN11-B --condition B --prompt {P} \
        --limit 300 --tag ch1prompt-tuned-{P}

## 7 · Package

In [ ]:
!zip -qr /kaggle/working/chapter1_results.zip results/
!du -sh /kaggle/working/chapter1_results.zip
!ls results/ | head -30

---
### Checklist

* [ ] tests pass (17/17)
* [ ] commit hash matches your push
* [ ] `transformers 4.57.6`, `torchao` absent
* [ ] every condition has **both** `acc_real` and `acc_anon`
* [ ] the `gap` column is populated for all six
* [ ] 3 seeds on **B vs C** — it carries the central claim
* [ ] every ranking caption says **50-way, filtered**
* [ ] `chapter1_results.zip` downloaded

> **A flat result is a result.** If C ≈ B, LoRA at 1.5B cannot install a type rule — and that is a finding about the method, not a failed experiment.